In [4]:
# ! pip install torch_geometric
# !pip install tensorflow
# !pip install keras
# !pip install zipfile
# !pip install torch-scatter -f https://pytorch-geometric.com/whl/torch-2.0.0+cu118.html

In [1]:
# 1 common_edge 以及rel做边

import os
os.chdir('/home/ec2-user/SageMaker/GNN_11_6/src')

In [ ]:
%run main.py --ATTENTION_TYPE "Naive" --want_TOP1 72.0 --path '/home/ec2-user/SageMaker/GNN_11_6' --epochs 5 --AA 1 --BB 1 --lambd 0 --drop_out 0.1 --scale_one_one 10 --res False --path_origin 'origin_sim'

load origin embedding!
2009
LEVEL1    TOP1 76.56%    TOP5 86.71%    TOP10 89.15%    TOP20 91.14%
SandR_Model(
  (R_Model): GATModel(
    (gat1): GATLayer(
      (gat_conv): GATConv(1536, 1024, heads=2)
      (batch_norm): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (activation): ReLU()
    )
    (gat2): GATLayer(
      (gat_conv): GATConv(2048, 1024, heads=3)
      (batch_norm): BatchNorm1d(3072, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (activation): ReLU()
    )
    (linear): Linear(in_features=3072, out_features=512, bias=True)
  )
)
------------------------Epoch: 001-----------------------
tensor([156.9837,  69.3657, 125.1728,  ...,   0.0000,   0.0000,   0.0000],
       grad_fn=<CopySlices>)
sppmi_pos_loss: 2419.657470703125
sppmi_neg_loss: 29908.017578125
Update rel!
batch_num: 000     LOSS: 1549.6060
MGB_REL:0.910985

In [9]:
import pandas as pd
import torch
import boto3
import numpy as np
def ret_type(codes, sep = ":"):
    """
    codes should be a list []
    """
    codes_split = [code.split(sep)[0] for code in codes]
    return(codes_split)


In [5]:
os.chdir('/home/ec2-user/SageMaker/GNN_11_6/src')
path = "2023-11-10 08:30:11"
emb = torch.load(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/rel_emb_1.pth')
name_all = pd.read_csv('https://han-attention.s3.amazonaws.com/input/name_desc/name_desc_all_GPT4.csv')
name_all = name_all['sign_id'].values
Final_emb = pd.DataFrame(emb.detach().numpy())
Final_emb.index = name_all
Final_emb.to_csv(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/rel_emb_naive_drop0.1_1111.csv')

s3 = boto3.client('s3')
with open(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/rel_emb_naive_drop0.1_1111.csv', 'rb') as f:
    s3.upload_fileobj(f, 'han-attention', f'input/phenotype_RA/rel_emb_naive_drop0.1_1111.csv')

In [6]:
index = np.sort(np.concatenate([np.where(np.array(ret_type(name_all)) == "RXNORM")[0], np.where(np.array(ret_type(name_all)) == "PheCode")[0]]))
name_all = name_all[index]

emb = emb[index,:]
Final_emb = pd.DataFrame(emb.detach().numpy())
Final_emb.index = name_all
Final_emb.to_csv(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/drug_side_emb_naive_drop0.1_1111.csv')

s3 = boto3.client('s3')
with open(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/drug_side_emb_naive_drop0.1_1111.csv', 'rb') as f:
    s3.upload_fileobj(f, 'han-attention', 'input/drug_side/drug_side_emb_naive_drop0.1_1111.csv')